In [ ]:
import astropy.visualization
import matplotlib.pyplot as plt
import named_arrays as na
import msfc_ccd

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
led = msfc_ccd.fits.open(msfc_ccd.samples.path_led_esis1)
dark = msfc_ccd.fits.open(msfc_ccd.samples.path_led_dark_esis1)

fig, ax = plt.subplots(
    figsize=(8, 4),
    constrained_layout=True,
)
im = na.plt.imshow(
    led.outputs.value,
    axis_x=led.axis_x,
    axis_y=led.axis_y,
    ax=ax,
)
ax.set_xlabel("detector $x$ (pix)")
ax.set_ylabel("detector $y$ (pix)")
plt.colorbar(im.ndarray.item(), ax=ax, label="signal (DN)");

In [ ]:
taps = led.taps
taps_dark = dark.taps

axis_x = taps.axis_x
axis_y = taps.axis_y
axis_tap_x = taps.axis_tap_x
axis_tap_y = taps.axis_tap_y

num_x = taps.num_x
num_blank = taps.camera.sensor.num_blank
num_overscan = taps.camera.sensor.num_overscan

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
na.plt.stairs(
    na.arange(0, num_blank + 1, axis=axis_x) - 0.5,
    taps_dark.unbiased.outputs[{axis_x: slice(0, num_blank)}].mean(axis_y),
    axis=axis_x,
    ax=ax,
    baseline=None,
)
na.plt.axvspan(
    xmin=num_blank - 25 - 0.5,
    xmax=num_blank - 0.5,
    color="green",
    alpha=0.2,
    ax=ax,
    label="columns used for the bias",
)
na.plt.set_ylim(-5, 5, ax=ax)
na.plt.set_ylabel("row-averaged signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("columns", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.95,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="right",
    va="top",
)
ax.ndarray.flat[0].legend(loc="lower right");

In [ ]:
signal = taps.unbiased.outputs - taps_dark.unbiased.outputs

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
num = 5
na.plt.stairs(
    na.arange(0, num_blank + num + 1, axis=axis_x) - 0.5,
    signal[{axis_x: slice(0, num_blank + num)}].mean(axis_y),
    axis=axis_x,
    ax=ax,
    baseline=None,
)
na.plt.axvspan(
    xmin=num_blank - 25 - 0.5,
    xmax=num_blank - 0.5,
    color="green",
    alpha=0.2,
    ax=ax,
    label="columns used for the bias",
)
na.plt.set_yscale("symlog", ax=ax, linthresh=1)
na.plt.set_ylabel("row-averaged signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("columns", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.95,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="top",
)
ax.ndarray.flat[0].legend(loc="center left");

In [ ]:
fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
num = 4
na.plt.stairs(
    na.arange(num_x - num_overscan - num, num_x + 1, axis=axis_x) - 0.5,
    signal[{axis_x: slice(-num_overscan - num, None)}].mean(axis_y),
    axis=axis_x,
    ax=ax,
    baseline=None,
)
na.plt.axvspan(
    xmin=num_x - num_overscan - 0.5,
    xmax=num_x - 0.5,
    color="red",
    alpha=0.2,
    ax=ax,
    label="overscan columns",
)
na.plt.set_yscale("log", ax=ax)
na.plt.set_ylabel("row-averaged signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("columns", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.05,
    y=0.05,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="left",
    va="bottom",
)
ax.ndarray.flat[0].legend();

In [ ]:
edge = signal[{axis_x: ~num_overscan}]
overscan_1 = signal[{axis_x: ~1}]
overscan_2 = signal[{axis_x: ~0}]

ratio_1 = overscan_1.mean(axis_y) / edge.mean(axis_y)
ratio_1

In [ ]:
ratio_2 = overscan_2.mean(axis_y) / edge.mean(axis_y)
ratio_2

In [ ]:
kwargs_filter = dict(
    size={axis_y: 51},
    proportion=0.05,
)
rows = na.arange(0, taps.num_y, axis=axis_y)

fig, ax = na.plt.subplots(
    axis_rows=axis_tap_y,
    axis_cols=axis_tap_x,
    nrows=taps.shape[axis_tap_y],
    ncols=taps.shape[axis_tap_x],
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    rows,
    na.ndfilters.trimmed_mean_filter(overscan_1, **kwargs_filter),
    axis=axis_y,
    ax=ax,
    label="first overscan column",
)
na.plt.plot(
    rows,
    na.ndfilters.trimmed_mean_filter(overscan_2, **kwargs_filter),
    axis=axis_y,
    ax=ax,
    label="second overscan column",
)
na.plt.plot(
    rows,
    ratio_1 * na.ndfilters.trimmed_mean_filter(edge, **kwargs_filter),
    axis=axis_y,
    ax=ax,
    color="black",
    linestyle="--",
    label="scaled last active column",
)
na.plt.set_ylabel("smoothed signal (DN)", ax[{axis_tap_x: 0}])
na.plt.set_xlabel("rows", ax=ax[{axis_tap_y: 0}])
na.plt.text(
    x=0.95,
    y=0.5,
    s=taps.label,
    ax=ax,
    transform=na.plt.transAxes(ax),
    ha="right",
    va="center",
)
handles, labels = ax.ndarray.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncols=3);

In [ ]:
kwargs_overscan = dict(num_blank=0, num_overscan=None)
taps.bias(**kwargs_overscan).outputs - taps_dark.bias(**kwargs_overscan).outputs

In [ ]:
taps.bias().outputs - taps_dark.bias().outputs

In [ ]:
taps_dark.unbiased.active.outputs.mean_trimmed(0.01, axis=taps.axis_xy)